# Lipi: Remote Benchmark Runner (Kaggle)

### Instructions:
1. **Set Accelerator**: In the right sidebar under **Notebook options**, set **Accelerator** to **GPU T4 x2** or **P100**.
2. **Turn Internet ON**: In the right sidebar, toggle **Internet** to **ON** (needed for code & Zenodo data).
3. **Set `PART`**: In Cell 1 below, change `PART = 1` (to `1`, `2`, `3`, ... `12`).
4. **Run All**: Press `Shift+Enter` or click **Run All**.
5. **Download**: When finished, download `lipi_part_XX.zip` from the right-hand **Output** panel.

In [ ]:
# =========================================================================
# CHANGE THIS NUMBER ONLY (1 to 12):
# =========================================================================
PART = 1   # Set to 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, or 12
# =========================================================================

TOTAL_PARTS = 12
WHO  = f"part_{PART:02d}"
WHAT = f"--shard {PART}/{TOTAL_PARTS}"
ZIP_NAME = f"lipi_part_{PART:02d}.zip"

REPO_URL = "https://github.com/ashimst/lipi.git"
DATA_URL = "https://zenodo.org/records/24240673/files/lipi_dataset_v1.0.zip?download=1"

In [ ]:
# -------------------------------------------------------------------------
# Setup Environment & Clone Repo
# -------------------------------------------------------------------------
import os, subprocess, sys
from pathlib import Path

if os.path.exists("/kaggle/working"):
    os.chdir("/kaggle/working")

if Path.cwd().name != "lipi":
    if not (Path.cwd() / "lipi").exists():
        print("Cloning repository...")
        subprocess.run(["git", "clone", REPO_URL, "lipi"], check=True)
    os.chdir("lipi")

subprocess.run(["git", "pull"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch
if torch.cuda.is_available():
    print(f"✓ GPU Ready: {torch.cuda.get_device_name(0)}")
else:
    print("⚠ WARNING: No GPU detected! Go to Kaggle sidebar -> Accelerator -> GPU T4 x2 or P100")

In [ ]:
# -------------------------------------------------------------------------
# Download & Verify Dataset (16,350 canonical images)
# -------------------------------------------------------------------------
if not os.path.exists("data/canonical"):
    kaggle_input = Path("/kaggle/input")
    found = list(kaggle_input.glob("**/canonical")) if kaggle_input.exists() else []
    
    if found:
        print(f"Found attached Kaggle dataset at: {found[0]}")
        os.makedirs("data", exist_ok=True)
        try:
            os.symlink(str(found[0]), "data/canonical")
        except OSError:
            import shutil
            shutil.copytree(str(found[0]), "data/canonical")
    elif DATA_URL:
        print(f"Downloading dataset from Zenodo ({DATA_URL})...")
        subprocess.run(["curl", "-L", "-o", "lipi_data.zip", DATA_URL], check=True)
        print("Extracting dataset into data/...")
        subprocess.run(["unzip", "-q", "lipi_data.zip", "-d", "data/"], check=True)
        if os.path.exists("lipi_data.zip"):
            os.remove("lipi_data.zip")

assert os.path.exists("data/canonical/brahmi"), (
    "ERROR: data/canonical not populated! Check that Internet access is toggled ON in the Kaggle sidebar."
)
print("✓ Dataset verified: 16,350 canonical glyph images available across 7 historical scripts.")

In [ ]:
# -------------------------------------------------------------------------
# Sanity Smoke Test
# -------------------------------------------------------------------------
!python run.py smoke

In [ ]:
# -------------------------------------------------------------------------
# Inspect & Run Assigned Shard Jobs
# -------------------------------------------------------------------------
print(f"=== ASSIGNMENT: PART {PART} (Jobs {WHAT}) ===\n")
!python run.py list {WHAT}
print(f"\nStarting training for Part {PART}...")
!python run.py train {WHAT} --who {WHO}

In [ ]:
# -------------------------------------------------------------------------
# Pack Everything (Results, Logs, Models) for Download
# -------------------------------------------------------------------------
out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
!python run.py pack --who {WHO} --name {ZIP_NAME} --out {out_dir}

target_zip = Path(out_dir) / ZIP_NAME
if target_zip.exists():
    size_mb = target_zip.stat().st_size / (1024 * 1024)
    print(f"\n" + "="*60)
    print(f"✓ SUCCESS: {ZIP_NAME} is ready ({size_mb:.1f} MB)!")
    print(f"  1. Download '{ZIP_NAME}' from the right-hand Kaggle Output pane.")
    print(f"  2. Drop '{ZIP_NAME}' into your local project's incoming/ folder.")
    print(f"  3. On your local machine, run: python run.py ingest")
    try:
        from IPython.display import FileLink
        display(FileLink(str(target_zip)))
    except Exception:
        pass
    print("="*60)
else:
    print("Warning: Output zip was not created.")